# Phase 3: three-model prediction ensemble (v8)

This notebook combines the three scored systems: the context BiLSTM incumbent (Codabench 0.4311), the safe low-rate continuation candidate (0.4278), and the selected five-fold CV `last_bilstm_ce` sequence model (0.3941). The candidate predictions match the saved v5 prediction exactly. It does not train or load model checkpoints; it combines saved hard-label predictions, so it runs quickly on CPU.

The context and candidate development means were 0.3671 and 0.4107. The CV sequence model's five-fold out-of-fold mean was 0.3696; that score comes from a different validation procedure and is not directly comparable. The context prediction set matches the 0.4311 incumbent. The candidate corresponds to the newer 0.4278 submission.

Because these saved files contain only final labels, not class probabilities, this uses a three-model majority vote. If all three models disagree, it keeps the context model's label. That is a conservative tie-break anchored to the strongest verified submission. Test labels are unavailable, so the ensemble's Codabench score cannot be predicted from these files.

Attach the existing Phase 3 dataset (for test JSON and `label_vocab.json`) and add `three_model_test_predictions_v8.json` as an input. This notebook validates the IDs, sentence counts, and labels, then creates one `submission.zip` with root-level `prediction.json`.


In [ ]:
import json, zipfile
from pathlib import Path
from collections import Counter

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/phase3_rr_v1_three_model_ensemble_v8")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def resolve_file(name):
    matches = sorted(p for p in INPUT_ROOT.rglob(name) if p.is_file())
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected exactly one {name!r} under /kaggle/input; found {matches}")
    return matches[0]

ASSETS = {name: resolve_file(name) for name in (
    "three_model_test_predictions_v8.json", "CL_test.json", "IT_test.json", "label_vocab.json")}
for name, path in ASSETS.items():
    print(f"{name}: {path}")


In [ ]:
with open(ASSETS["three_model_test_predictions_v8.json"], "r", encoding="utf-8") as f:
    bundle = json.load(f)
with open(ASSETS["label_vocab.json"], "r", encoding="utf-8") as f:
    label_to_id = json.load(f)
allowed_labels = set(label_to_id)
TEST = {}
for domain in ("CL", "IT"):
    with open(ASSETS[f"{domain}_test.json"], "r", encoding="utf-8") as f:
        TEST[domain] = json.load(f)

MODEL_NAMES = ("context", "candidate", "sequence")
if set(bundle.get("predictions", {})) != {"CL", "IT"}:
    raise ValueError("Prediction bundle must contain CL and IT domains")
for domain in ("CL", "IT"):
    outputs = bundle["predictions"][domain]
    if set(outputs) != set(MODEL_NAMES):
        raise ValueError(f"{domain}: expected predictions from exactly {MODEL_NAMES}")
    expected_ids = set(map(str, TEST[domain]))
    for name in MODEL_NAMES:
        rows = outputs[name]
        if not isinstance(rows, list):
            raise TypeError(f"{domain}/{name}: expected list of document predictions")
        mapping = {}
        for row in rows:
            doc_id = str(row["id"])
            if doc_id in mapping:
                raise ValueError(f"{domain}/{name}: duplicate document {doc_id}")
            mapping[doc_id] = row["predictions"]
        if set(mapping) != expected_ids:
            raise ValueError(f"{domain}/{name}: document IDs differ from official test data")
        for doc_id, sentences in TEST[domain].items():
            labels = mapping[str(doc_id)]
            if len(labels) != len(sentences):
                raise ValueError(f"{domain}/{name}/{doc_id}: sentence count mismatch")
            if any(label not in allowed_labels for label in labels):
                raise ValueError(f"{domain}/{name}/{doc_id}: unknown label")
        outputs[name] = mapping
print("All three prediction sets match the official test IDs, sentence counts, and label vocabulary.")


In [ ]:
prediction = {}
by_domain = {}
audit = {"domains": {}, "models": bundle["models"], "tie_rule": bundle["tie_rule"]}
for domain in ("CL", "IT"):
    context = bundle["predictions"][domain]["context"]
    candidate = bundle["predictions"][domain]["candidate"]
    sequence = bundle["predictions"][domain]["sequence"]
    rows = []
    total = unanimous = majority = all_different = changed_vs_context = 0
    pairwise = Counter()
    for doc_id, sentences in TEST[domain].items():
        doc_id = str(doc_id)
        merged = []
        for c, n, q in zip(context[doc_id], candidate[doc_id], sequence[doc_id]):
            total += 1
            pairwise["context_candidate"] += int(c != n)
            pairwise["context_sequence"] += int(c != q)
            pairwise["candidate_sequence"] += int(n != q)
            if c == n == q:
                unanimous += 1
                chosen = c
            else:
                counts = Counter((c, n, q))
                top = max(counts.values())
                if top >= 2:
                    majority += 1
                    chosen = next(label for label, count in counts.items() if count == top)
                else:
                    all_different += 1
                    chosen = c
            changed_vs_context += int(chosen != c)
            merged.append(chosen)
        prediction[doc_id] = merged
        rows.append({"id": doc_id, "predictions": merged})
    by_domain[domain] = rows
    audit["domains"][domain] = {
        "documents": len(rows), "sentences": total,
        "unanimous": unanimous, "majority_decisions": majority,
        "all_different_ties_context_wins": all_different,
        "changed_from_context": changed_vs_context,
        "pairwise_disagreements": dict(pairwise)}

if len(prediction) != sum(len(TEST[d]) for d in ("CL", "IT")):
    raise ValueError("Duplicate document IDs across CL and IT")


In [ ]:
prediction_path = OUTPUT_DIR / "prediction.json"
with open(prediction_path, "w", encoding="utf-8") as f:
    json.dump(prediction, f, ensure_ascii=False, indent=2)
with open(OUTPUT_DIR / "three_model_ensemble_audit.json", "w", encoding="utf-8") as f:
    json.dump(audit, f, ensure_ascii=False, indent=2)
with open(OUTPUT_DIR / "three_model_ensemble_by_domain.json", "w", encoding="utf-8") as f:
    json.dump(by_domain, f, ensure_ascii=False, indent=2)

zip_path = OUTPUT_DIR / "submission.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(prediction_path, arcname="prediction.json")
with zipfile.ZipFile(zip_path, "r") as zf:
    if zf.namelist() != ["prediction.json"]:
        raise ValueError(f"Unexpected ZIP members: {zf.namelist()}")
    if zf.testzip() is not None:
        raise ValueError("ZIP CRC check failed")
    packaged = json.loads(zf.read("prediction.json").decode("utf-8"))
if packaged != prediction:
    raise ValueError("ZIP payload differs from prediction.json")

print("Output ZIP:", zip_path)
print(json.dumps(audit["domains"], indent=2))
print("PASS: one root-level prediction.json with aligned documents and label counts.")


## Interpretation

The vote is based only on the three saved model outputs. It is not a newly trained model, and it does not use hidden test labels. The audit reports agreement and how often the ensemble changes the context incumbent; these counts describe model disagreement, not correctness. Compare the eventual Codabench score with the preserved 0.4311 submission before replacing it as the best result.
